# Cross-fit XGBoost-PU 配置参数实验

本 notebook 专门做**配置受控搜索**，回答「哪一种 XGBoost-PU 配置在固定 90% 样本外召回下给出的候选最少、最稳」。

搜索分三段，**所有配置共享完全相同的折分与随机种子**，因此配置之间的差异不是划分噪声：

| 阶段 | 变量 | 规模 |
| --- | --- | --- |
| 1. 比例预实验 | `U:P = 1:1 / 3:1 / 5:1 / 10:1` | 5 折 × 1 重复 × 30 bag |
| 2. 正则化预实验 | 前 2 优比例 × `max_depth∈{2,3}` × `min_child_weight∈{1,3}` × `reg_lambda∈{1,5}` | 5 折 × 1 重复 × 30 bag |
| 3. 正式复验 | 综合最优 3 组配置 | 5 折 × 3 重复 × 100 bag |

模型选择规则（顺序不可颠倒）：

1. 已知星 OOF 召回必须 >= 82/91；
2. 满足召回的配置中 `N_candidates @ Recall90` 最少；
3. 候选数量与阈值的三次重复波动更小者优先；
4. 结构与参数更简单者优先。

**不允许**用固定 top-N、事后手工挪阈值，或用文献预期数量（DR5 约 100 颗、DR13 预期 100–400 颗）反推配置。

本 notebook 只导出 `config_search_summary.csv` / `.json` 与清单，不导出每个配置的全量分数表；所有中间变量都保留在命名空间中，方便你后期自己接着做可视化。

运行环境：`D:\Anaconda\envs\myenv\python.exe`

## 0. 环境与路径

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass


def find_project_root(start):
    """Locate the repository root by its sentinel module."""
    for candidate in [start, *start.parents]:
        if (candidate / "build_dr13_all_cache.py").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from crossfit.crossfit_engine import (
    BEST_FORMAL_CONFIG,
    recall_tag,
    load_project_data,
    run_formal_line,
)
from crossfit import crossfit_plotting as cplot

RESULTS_DIR = PROJECT_ROOT / "crossfit" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cplot.apply_style()

TOP_N = 30          # exported spectrum figure shows this many candidates
QUICK = False       # set True for a fast smoke run (3 folds, 1 repeat, 2 bags)

print("Python      :", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Results dir :", RESULTS_DIR.resolve())
print("Best config :", BEST_FORMAL_CONFIG)

## 1. 数据与折分

In [ ]:
# The cache is read once and reused by every line below.
data = load_project_data()
common_wave = np.asarray(data["common_wave"])
stars = pd.DataFrame(data["stars_clean"]).reset_index(drop=True)

summary_table = pd.DataFrame(
    {
        "quantity": [
            "spectra",
            "wavelength pixels",
            "wavelength range (Angstrom)",
            "known CN stars (label=1)",
            "unlabeled objects (label=-1)",
            "positive fraction",
        ],
        "value": [
            len(stars),
            common_wave.size,
            f"{common_wave.min():.0f}-{common_wave.max():.0f}",
            int((stars["label"] == 1).sum()),
            int((stars["label"] == -1).sum()),
            f"1 : {(stars['label'] == -1).sum() / max((stars['label'] == 1).sum(), 1):.0f}",
        ],
    }
)
display(summary_table)
print("X_clean shape:", np.asarray(data["X_clean"]).shape)

In [ ]:
from crossfit.crossfit_engine import CrossFitConfig, make_fixed_folds, label_array

y = label_array(stars)
fold_probe = CrossFitConfig(n_splits=5, n_repeats=3, n_bags=100, target_recall=0.90, random_seed=42)
fixed_folds = make_fixed_folds(y, fold_probe)

fold_table = pd.DataFrame(
    [
        {
            "repeat": r + 1,
            "fold": f + 1,
            "n_train": int(train_idx.size),
            "n_train_positive": int(y[train_idx].sum()),
            "n_holdout": int(holdout_idx.size),
            "n_holdout_positive": int(y[holdout_idx].sum()),
        }
        for r, repeat_folds in enumerate(fixed_folds)
        for f, (train_idx, holdout_idx) in enumerate(repeat_folds)
    ]
)
display(fold_table.head(15))
print("folds per repeat:", [len(rep) for rep in fixed_folds])
print("every object held out exactly once per repeat:", all(
    sorted(np.concatenate([h for _, h in rep]).tolist()) == list(range(len(y))) for rep in fixed_folds
))

## 2. 运行配置搜索

`SMOKE = True` 时用 3 折 × 1–2 重复 × 2–3 bag 快速跑通流程，仅用于验证代码；正式结论必须用 `SMOKE = False`。

In [ ]:
from crossfit.crossfit_engine import run_config_search

SMOKE = False
SEARCH_DIR = RESULTS_DIR / ("search_smoke" if SMOKE else "search")
SEARCH_DIR.mkdir(parents=True, exist_ok=True)

bundle = run_config_search(SEARCH_DIR, progress=True, smoke=SMOKE, data=data)

search_summary = bundle["search_summary"]
stage_results = bundle["stage_results"]
formal_results = bundle["formal_results"]
best_config = bundle["best_config"]
best_result = bundle["best_result"]
best_tables = bundle["best_tables"]
X_model = bundle["X_model"]

print()
print("search rows      :", len(search_summary))
print("stage counts     :", bundle["search_summary_meta"]["stage_counts"])
print("best config key  :", bundle["search_summary_meta"]["best_config_key"])
print("best config      :", best_config)

## 3. 搜索结果总表

`candidate_count` 是固定 90% 样本外召回下的未标注候选数量，越小越好；`candidate_count_std` / `threshold_std` 是三次重复的波动。

In [ ]:
display(
    search_summary.sort_values(["stage", "candidate_count"]).reset_index(drop=True)
)

## 4. 配置对比可视化

In [ ]:
cplot.plot_config_search(search_summary)

## 5. 最优配置的正式结果

In [ ]:
print("best config key:", best_result["config_key"])
print("threshold      :", round(float(best_result["threshold"]), 6))
print(
    "known recall   :",
    f"{best_result['achieved_known_recall']:.4f}",
    f"({best_result['required_known']}/{len(best_tables['known_cn'])} retained)",
)
print("candidates     :", f"{len(best_tables['candidates']):,}")
print("repeat thresholds       :", np.round(np.asarray(best_result["repeat_thresholds"]), 6).tolist())
print("repeat candidate counts :", np.asarray(best_result["repeat_candidate_counts"]).tolist())

cplot.plot_score_distribution(best_tables["all_scores"], best_result)
cplot.plot_repeat_stability(best_result)

## 6. 正式复验三组配置的对比

这里展示三组进入正式复验的配置在重复间的一致性——这是判断「最优」是否稳健的关键。

In [ ]:
formal_keys = list(formal_results.keys())
formal_rows = []
for key, entry in formal_results.items():
    result = entry["result"]
    tables = entry["tables"]
    formal_rows.append(
        {
            "config_key": key,
            "threshold": float(result["threshold"]),
            "candidates": len(tables["candidates"]),
            "candidate_count_std": float(np.std(result["repeat_candidate_counts"], ddof=1)),
            "recall": float(result["achieved_known_recall"]),
            "roc_auc": float(result["roc_auc"]),
            "pr_auc": float(result["pr_auc"]),
        }
    )
formal_frame = pd.DataFrame(formal_rows).sort_values("candidates")
display(formal_frame.reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for key, entry in formal_results.items():
    scores = entry["result"]["score_mean"]
    labels = entry["tables"]["all_scores"]["label"].to_numpy()
    axes[0].hist(scores[labels == -1], bins=80, histtype="step", linewidth=1.4, density=True, label=key)
    axes[1].plot(
        np.arange(1, len(entry["tables"]["candidates"]) + 1),
        np.sort(entry["tables"]["candidates"]["xgb_pu_score"])[::-1],
        linewidth=1.4,
        label=key,
    )
axes[0].set_yscale("log")
axes[0].set_xlabel("OOF xgb_pu_score")
axes[0].set_ylabel("density (log)")
axes[0].set_title("Unlabeled score distribution by configuration")
axes[0].legend(fontsize=8)
axes[1].set_xlabel("Candidate rank")
axes[1].set_ylabel("OOF xgb_pu_score")
axes[1].set_title("Candidate score decay by configuration")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. 保留的变量（供后期自定义可视化）

跑完上面的单元后，以下变量都在命名空间里，可直接用：

| 变量 | 内容 |
| --- | --- |
| `data` / `stars` / `common_wave` | 原始缓存、星表（含 label/参数）与波长轴 |
| `y` / `X_model` / `fixed_folds` | PU 标签、标准化后的 700 维特征矩阵、固定折分 |
| `search_summary` | 全部配置的阶段、候选数、阈值、稳定性总表 |
| `stage_results` | 每个配置的 `config / result / tables`（含 41,243 行 `all_scores`） |
| `formal_results` | 三组正式复验的完整结果 |
| `best_result` / `best_tables` / `best_config` | 最优配置及其候选表 |

常用派生示例：某配置的候选掩码、任意两个配置候选名单的交集、按分数分位切层、把候选与 Teff/logg/[Fe/H] 联合作图等。

In [ ]:
print("Retained variables")
for name, value in [
    ("data", data),
    ("stars", stars),
    ("common_wave", common_wave),
    ("y", y),
    ("X_model", X_model),
    ("fixed_folds", fixed_folds),
    ("search_summary", search_summary),
    ("stage_results", stage_results),
    ("formal_results", formal_results),
    ("best_config", best_config),
    ("best_result", best_result),
    ("best_tables", best_tables),
]:
    shape = getattr(value, "shape", None)
    print(f"  {name:16s} {type(value).__name__:12s} {shape if shape is not None else ''}")
print()
print("stage_results keys (first 5):", list(stage_results)[:5])
print("all_scores columns:", list(best_tables["all_scores"].columns))

### 示例：自定义可视化——最优配置候选在 Teff–logg 平面的密度与已知星位置

In [ ]:
custom = best_tables["all_scores"]
cand = best_tables["candidates"]
u_part = custom[custom["label"] == -1]
known_part = custom[custom["label"] == 1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].hexbin(u_part["teff"], u_part["logg"], gridsize=45, cmap="Greys", mincnt=1)
axes[0].scatter(known_part["teff"], known_part["logg"], s=30, color="darkorange", label="known CN")
axes[0].scatter(cand["teff"], cand["logg"], s=10, color="crimson", alpha=0.7, label=f"candidates ({len(cand):,})")
axes[0].set_xlabel("Teff")
axes[0].set_ylabel("logg")
axes[0].set_title("Candidates on the Teff-logg plane")
axes[0].legend()

sc = axes[1].scatter(
    cand["teff"], cand["feh"], c=cand["xgb_pu_score"], s=14, cmap="magma_r", alpha=0.85
)
axes[1].scatter(known_part["teff"], known_part["feh"], s=30, facecolors="none", edgecolors="steelblue", label="known CN")
axes[1].set_xlabel("Teff")
axes[1].set_ylabel("[Fe/H]")
axes[1].set_title("Candidate score coloured by [Fe/H]")
fig.colorbar(sc, ax=axes[1], label="xgb_pu_score")
axes[1].legend()
plt.tight_layout()
plt.show()

### 示例：任意两个配置的候选交集

In [ ]:
keys = list(formal_results)[:2]
if len(keys) == 2:
    left = set(formal_results[keys[0]]["tables"]["candidates"]["uid"].astype(str))
    right = set(formal_results[keys[1]]["tables"]["candidates"]["uid"].astype(str))
    print("config A:", keys[0], "candidates:", len(left))
    print("config B:", keys[1], "candidates:", len(right))
    print("intersection:", len(left & right))
    print("A only:", len(left - right), " B only:", len(right - left))
    print("Jaccard:", round(len(left & right) / max(len(left | right), 1), 4))

## 8. 导出清单

In [ ]:
files = sorted(p.relative_to(SEARCH_DIR).as_posix() for p in SEARCH_DIR.rglob("*") if p.is_file())
display(pd.DataFrame({"exported file": files}))
print("search dir:", SEARCH_DIR.resolve())
print()
print("Search stage exports only the search table and its summary;")
print("the full all-score tables stay in memory as notebook variables.")

### 使用建议

- 若正式复验三组配置的 `candidate_count_std` 与 `threshold_std` 都很大，说明 91 颗正例不足以稳定标定阈值，此时应优先考虑：增加重复次数与 bag 数、或把「多次重复都被选中」的交集作为高置信核心集，而不是继续细调超参数。
- 预实验仅用于筛掉明显更差的配置，**最终结论只引用正式复验**。
- 把 `best_config` 与 `BEST_FORMAL_CONFIG`（引擎默认值）保持一致，主结果 notebook 才会使用同一配置。